# Forecast ensemble time series and initial shock

## Purpose

Follow every hindcast member from the initial state (lead 0) through the forecast, verify the regional series against observations, and diagnose the adjustment right after initialization: the regional mean absolute tendency |dX/dt| against its settled level and against the reference, the ensemble-mean bias drift, RMSE and spread growth, and the members' spatial extremes. A summary table condenses each experiment's shock into a few numbers. For pairwise experiment-minus-baseline decay fits see `06_compute_and_plot_decay_timescales.ipynb`.

## Inputs

Post-processed member time series `ts/<frequency>/<variable>.EN##.<year>.nc` of the hindcasts registered in `util/bias_metrics.py` (S0: Jan-Feb 2012 from the Dec 2011 analyses; S1: Jun-Jul 2012 from the May 2012 analyses), and references under `OBSERVATION_ROOT`: ERA5 6-hourly (PSL, TS, TREFHT, PRECT), GPM IMERG daily (PRECT), NOAA interpolated OLR daily (FLUT).

## Modes

The user setup chooses what is plotted: `SEASON` (S0 or S1), the `EXPERIMENTS` compared, and the `REGIONS`. `MODE` selects one entry of `MODES`, which sets the kind of view: output frequency, variables, references, member display, plotted lead range, and figure kinds.

- **`shock`**: 6-hourly output resolves the first-day adjustment; PSL, TS, TREFHT, PRECT (ERA5) and FLUT (NOAA OLR); individual members over the first 10 days.
- **`daily_obs`**: daily means against observations (PRECT vs GPM IMERG, FLUT vs NOAA OLR), with the 25-75% member envelope over the whole forecast (pair it with regional boxes such as CONUS or NHMidLat).

## Processing

For each variable and experiment the driver reads every member, places each record at its lead time, averages the reference over the same intervals, and reduces to regional diagnostics. Results are cached per experiment/variable/frequency and reused unless they are missing, built with another ensemble size, reference, or cache version, or lack a requested region; `FORCE_COMPUTE = True` rebuilds them.

## Outputs

NetCDF caches and the shock-summary CSV under `DIAGNOSTIC_OUTPUT_ROOT/data/fcst_atm_ensemble_ts/`; figures under `DIAGNOSTIC_OUTPUT_ROOT/figure/fcst_atm_ensemble_ts/<MODE>/`, named by variable, region, and season. Review the user setup, then run the notebook from top to bottom.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## 2. User setup

Paths; the season, experiments, and regions to plot; the kind of view (`MODE`); shock windows; and run options. Edit this cell for normal changes.

In [2]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")  # land mask
WORKFLOW_NAME = "fcst_atm_ensemble_ts"

# What to plot
SEASON = "S0"  # "S0": Jan-Feb 2012 hindcasts; "S1": Jun-Jul 2012 hindcasts
# S0: CTRL10-S0, CAPT10-S0, DART10-S0, DART20-S0, DART40-S0
# S1: CTRL10-S1, CAPT10-S1, DART40-S1
EXPERIMENTS = ("CTRL10-S0", "CAPT10-S0", "DART10-S0", "DART20-S0", "DART40-S0")
# global, land, ocean, Tropics, CONUS, NHMidLat, NINO3.4, MAR_CONT
REGIONS = ("global", "land", "ocean", "Tropics")

# Kind of view. References: PSL/TS/TREFHT "ERA5"; PRECT "ERA5" or "GPM";
# FLUT "NOAA OLR". member_display: "members", "quartiles", or "none".
# plots: "timeseries" (members, mean, reference) and/or "shock" (diagnostics).
MODES = {
    "shock": dict(
        frequency="6hourly",
        variables=("PSL", "TS", "TREFHT", "PRECT", "FLUT"),
        references={"PRECT": "ERA5"},
        member_display="members",
        zoom_days=10,
        plots=("timeseries", "shock"),
    ),
    "daily_obs": dict(
        frequency="daily",
        variables=("PRECT", "FLUT"),
        references={"PRECT": "GPM"},
        member_display="quartiles",
        zoom_days=60,
        plots=("timeseries",),
    ),
}
MODE = "shock"

# Shock diagnostics (lead days)
SHOCK_DAYS = 2.0  # window treated as the initial shock
SETTLED_DAYS = (10.0, 30.0)  # window taken as the adjusted state
TOLERANCE = 0.2  # adjustment lasts while the 1-day running |dX/dt| exceeds (1 + TOLERANCE) x settled

USE_REFERENCE = True  # False: model-only diagnostics
MAX_MEMBERS = None  # e.g. 2 for a quick test; caches record the ensemble size
FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 200

### Mode selection

Set `MODE = "shock"` for the 6-hourly initial-shock view or `MODE = "daily_obs"` for daily series against observations, or add an entry to `MODES`. `SEASON`, `EXPERIMENTS`, and `REGIONS` apply to every mode. Caches are per experiment, variable, and frequency, so modes with the same frequency share them; a cache is rebuilt when it lacks a requested region or was built with a different reference.

## 3. Driver: read, process, and save data

Reuse valid caches or build missing ones, then compute and save the shock summary.

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.forecast_ensemble_ts import read_or_process_ensemble_ts, shock_summary_table, validate_mode, validate_regions

config = validate_mode(MODE, MODES)
validate_regions(REGIONS)
data_dir, figure_root = workflow_output_dirs(WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT)
figure_dir = figure_root / MODE
print(f"{SEASON} {list(EXPERIMENTS)} | regions {list(REGIONS)} | mode {MODE}: {config}")

results, registries, failures = read_or_process_ensemble_ts(
    {SEASON: tuple(EXPERIMENTS)}, config["variables"], data_root=INPUT_DATA_ROOT, cache_dir=data_dir,
    frequency=config["frequency"], regions=REGIONS,
    landmask_file=LAND_SUPPORT_DIR / "landmask_1x1.nc",
    reference_root=OBSERVATION_ROOT if USE_REFERENCE else None,
    references=config["references"], max_members=MAX_MEMBERS, force_compute=FORCE_COMPUTE,
)
if failures:
    print(f"\n{len(failures)} case(s) failed and are skipped below:")
    for season, variable, experiment, error in failures:
        print(f"  {season} {variable} {experiment}: {error}")

summary = shock_summary_table(results, shock_days=SHOCK_DAYS, settled_days=SETTLED_DAYS,
                              tolerance=TOLERANCE)
table_path = data_dir / f"shock_summary_{SEASON}_fc_{MODE}.csv"
summary.to_csv(table_path, index=False)
print(f"[TABLE] {table_path}")

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


S0 ['CTRL10-S0', 'CAPT10-S0', 'DART10-S0', 'DART20-S0', 'DART40-S0'] | regions ['global', 'land', 'ocean', 'Tropics'] | mode shock: {'references': {'PRECT': 'ERA5'}, 'member_display': 'members', 'zoom_days': 10, 'plots': ('timeseries', 'shock'), 'frequency': '6hourly', 'variables': ('PSL', 'TS', 'TREFHT', 'PRECT', 'FLUT')}


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


[WRITE] CTRL10-S0_PSL_6hourly_201201-201202_ensemble_ts.nc (cache version 1, current 2)


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/dask/array/core.py:5097: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(


## 4. Plot and show

Thick lines are ensemble means and dashed black the reference; the shock diagnostics show tendency, bias drift, RMSE (solid) and spread (dotted), and spatial extremes.

In [ ]:
from util.forecast_ensemble_ts import plot_ensemble_ts_figures, plot_shock_summary_figures, results_colors

colors = results_colors(results)
figures = plot_ensemble_ts_figures(
    results, registries, regions=REGIONS, figure_dir=figure_dir, tag=MODE,
    plots=config["plots"], colors=colors, zoom_days=config["zoom_days"],
    member_display=config["member_display"], dpi=FIGURE_DPI, show=SHOW,
)

### Shock summary

One row per variable, experiment, and region; grouped bars per region.

In [ ]:
display(summary.round(3))
summary_figures = plot_shock_summary_figures(
    summary, figure_dir=figure_dir, tag=MODE, colors=colors, dpi=FIGURE_DPI, show=SHOW,
)

## Notes: lead time, diagnostics, and summary metrics

**Lead time.** Each hindcast is one continuous run from the analysis time. The first model record has zero width and is the initial state (lead 0). Later records are placed at the midpoint of their output interval for interval means (`time: mean`, e.g. TS, TREFHT, PRECT) and at the interval end for instantaneous values (`time: point`, e.g. PSL, PS). For flux and process variables (e.g. PRECT) the lead-0 value is the first model step, not a balanced state, so their `initial_bias` and first tendency mostly show model spin-up.

**Reference.** On the same 180x360 grid. `PSL`, `TS`, `TREFHT`, `PRECT`: ERA5 6-hourly samples averaged with trapezoid weights over each model interval (the sample at that time for lead 0 and instantaneous variables); for 6-hourly model means this is the mean of the two bounding samples, an approximation. `FLUT`: NOAA interpolated OLR daily means, overlap-weighted over each model interval. At 6-hourly frequency each FLUT interval is compared with its day's mean, so the bias and RMSE include the model's diurnal OLR cycle, lead 0 has no reference, and the reference tendency is left missing (daily values have no within-day change); with daily output (`frequency="daily"`) the comparison is exact. `PRECT` can instead use GPM IMERG daily means (`references={"PRECT": "GPM"}` in a mode), handled the same way; IMERG covers about 60S-60N, and when a reference leaves gridpoints missing every diagnostic of that variable uses only the covered gridpoints, so model and reference describe the same area. Other variables in `ts/<frequency>/` (e.g. `PS`, `TMQ`) give model-only diagnostics.

| Diagnostic | Definition |
|---|---|
| `member_mean`, `member_min`, `member_max` | cos(lat)-weighted regional mean, and regional min/max, per member |
| `member_tendency` | regional mean of the gridpoint \|X(t) - X(t-1)\| / dt, per member (units per day); for interval means it starts at the first mean, skipping the step from the lead-0 snapshot |
| `ensemble_spread` | regional mean of the gridpoint ensemble standard deviation |
| `reference_mean`, `reference_tendency` | the same for the reference (tendency missing when the reference is coarser than the model output) |
| `ensemble_mean_rmse` | gridpoint RMSE of the ensemble mean against the reference over the region |

| Summary metric | Definition |
|---|---|
| `tendency_ratio` | ensemble-mean \|dX/dt\| over leads (0, `SHOCK_DAYS`] divided by its mean over `SETTLED_DAYS`; > 1 means the model adjusts faster right after initialization than once settled |
| `tendency_ratio_ref` | the same shock-window tendency divided by the reference's over the same dates; separates the shock from the observed weather |
| `adjustment_days` | last lead before `SETTLED_DAYS` where the 1-day running \|dX/dt\| exceeds (1 + `TOLERANCE`) x the settled level; 0 = no detectable shock |
| `initial_bias` | ensemble-mean minus reference regional mean at the first lead with a reference (lead 0 for ERA5 variables: the analysis error) |
| `drift` | bias change from that lead to the last day of the shock window |
| `spread_ratio` | ensemble spread on the last day of the shock window divided by the spread at lead 0 |

The 6-hourly tendency includes the diurnal cycle and atmospheric tides; both the settled level and the 6-hourly ERA5 reference contain them too, so the ratios compare like with like. Absolute tendencies of instantaneous and interval-mean variables are not comparable with each other.

Caches: `<experiment>_<variable>_<frequency>_<period>_ensemble_ts.nc` (dims member, lead, region; the reference product and a cache version are file attributes, and caches from an older version are rebuilt); table: `shock_summary_<SEASON>_fc_<MODE>.csv`. The `daily_obs` mode replaces the former regional time-series notebook.